In [17]:
import numpy as np

def continuous_kronecker_prod(matrices):
    """
    Compute the continuous Kronecker sum of a list of matrices.
    
    Parameters:
    - matrices: list of numpy arrays, where each array is a square matrix.
    
    Returns:
    - The resulting Kronecker sum matrix.
    """
    if len(matrices) < 2:
        return np.array(matrices[0])
    
    # Start with the first two matrices
    result = np.kron(matrices[0], matrices[1])
    
    # Iterate through the rest of the matrices
    for i in range(2, len(matrices)):
        result = np.kron(result, matrices[i])
    
    return result

def continuous_kron_sum(matrices):
    """
    Compute the continuous Kronecker sum of a list of matrices.
    
    Parameters:
    - matrices: list of numpy arrays, where each array is a square matrix.
    
    Returns:
    - The resulting Kronecker sum matrix.
    """
    if len(matrices) < 2:
        return np.array(matrices[0])
    
    # Helper function for single Kronecker sum
    def kron_sum(A, B):
        a = A.shape[0]  # Order of matrix A
        b = B.shape[0]  # Order of matrix B
        I_a = np.eye(a)
        I_b = np.eye(b)
        return np.kron(A, I_b) + np.kron(I_a, B)
    
    # Start with the first two matrices
    result = kron_sum(matrices[0], matrices[1])
    
    # Iterate through the rest of the matrices
    for i in range(2, len(matrices)):
        result = kron_sum(result, matrices[i])
    
    return result


# Inner function: Merge close values
def clean_set(set, threshold=1e-4, integer_threshold=1e-7):
    # Sort set for easier merging of close values
    sorted_set = sorted(set)
    
    # Store the merged results
    merged_set = []
    
    # Traverse each value and merge close values
    for value in sorted_set:
        # If merged_set is empty or the difference between the current value and the last added value is greater than the threshold, add directly
        if not merged_set or abs(value - merged_set[-1]) > threshold:
            # If the current value is close to an integer, replace it with the nearest integer
            if np.isclose(value, round(value), atol=integer_threshold):
                value = round(value)  # Replace with the nearest integer
            merged_set.append(value)
    
    return merged_set

def compute_omegas(eigs):
    list_omega = [round(eig - eigs[0]) for eig in eigs]  # Compute the differences
    list_omega = [omega for omega in list_omega if omega > 0]  # Filter positive values
    return list_omega

X = np.array([[0, 1], [1, 0]])
Y = np.array([[0, -1j], [1j, 0]])
Z = np.array([[1, 0], [0, -1]])
I = np.eye(2)






matrices = [X,X,X,X]  # Three Pauli X matrices

result = continuous_kron_sum(matrices)
print(result)

EIGS,EIGVS=np.linalg.eigh(result)
print(EIGS[-1])
print(EIGVS[:,-1])

[[0. 1. 1. 0. 1. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
 [1. 0. 0. 1. 0. 1. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [1. 0. 0. 1. 0. 0. 1. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 1. 1. 0. 0. 0. 0. 1. 0. 0. 0. 1. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 1. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 1. 0. 0. 1. 0. 0. 1. 0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 1. 0. 1. 0. 0. 1. 0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 1. 0. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 0. 1. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 1. 0. 0. 1. 0. 1. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0. 1. 0. 0. 1. 0. 0. 1. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 1. 1. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 1. 0. 0. 0. 1. 0. 0. 0. 0. 1. 1. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0. 0. 1. 0. 0. 1. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 1. 0. 1. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 1. 0. 1. 1. 0.]]
3.9999999999999996
[0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25
 0.25 0.25]


In [18]:
def H_TWO_ROTATIONS(num_q, pauli=Z, periodic_boundary=True):
    result = np.zeros((2**num_q, 2**num_q), dtype=complex)

    for i in range(num_q-1):
        box = [I for _ in range(num_q)]
        box[i] = pauli
        box[i+1] = pauli

        result += continuous_kronecker_prod(box)


    if periodic_boundary:
        box = [I for _ in range(num_q)]
        box[0] = pauli
        box[num_q-1] = pauli
        result += continuous_kronecker_prod(box)

    result = -0.5 * result            

    EIGS,EIGVS = np.linalg.eigh(result)

    return EIGS,EIGVS

In [19]:
for i in range(2,12):
    EIGS,EIGVS = H_TWO_ROTATIONS(i,pauli=Y, periodic_boundary=True)
    print(i,compute_omegas(clean_set(EIGS)))

2 [2]
3 [2]
4 [2, 4]
5 [2, 4]
6 [2, 4, 6]
7 [2, 4, 6]
8 [2, 4, 6, 8]
9 [2, 4, 6, 8]
10 [2, 4, 6, 8, 10]
11 [2, 4, 6, 8, 10]


In [20]:
def H_ONE_ROTATIONS(num_q, pauli=Z):
    box = [pauli for _ in range(num_q)]
    result = -0.5 * continuous_kron_sum(box)    
    EIGS,EIGVS = np.linalg.eigh(result)

    return EIGS,EIGVS

for i in range(2,12):
    EIGS,EIGVS = H_ONE_ROTATIONS(i,pauli=X)
    print(i,compute_omegas(clean_set(EIGS)))

# for i in range(2,12):
    EIGS,EIGVS = H_ONE_ROTATIONS(i,pauli=Y)
    print(i,compute_omegas(clean_set(EIGS)))

# for i in range(2,12):
    EIGS,EIGVS = H_ONE_ROTATIONS(i,pauli=Z)
    print(i,compute_omegas(clean_set(EIGS)))

2 [1, 2]
2 [1, 2]
2 [1, 2]
3 [1, 2, 3]
3 [1, 2, 3]
3 [1, 2, 3]
4 [1, 2, 3, 4]
4 [1, 2, 3, 4]
4 [1, 2, 3, 4]
5 [1, 2, 3, 4, 5]
5 [1, 2, 3, 4, 5]
5 [1, 2, 3, 4, 5]
6 [1, 2, 3, 4, 5, 6]
6 [1, 2, 3, 4, 5, 6]
6 [1, 2, 3, 4, 5, 6]
7 [1, 2, 3, 4, 5, 6, 7]
7 [1, 2, 3, 4, 5, 6, 7]
7 [1, 2, 3, 4, 5, 6, 7]
8 [1, 2, 3, 4, 5, 6, 7, 8]
8 [1, 2, 3, 4, 5, 6, 7, 8]
8 [1, 2, 3, 4, 5, 6, 7, 8]
9 [1, 2, 3, 4, 5, 6, 7, 8, 9]
9 [1, 2, 3, 4, 5, 6, 7, 8, 9]
9 [1, 2, 3, 4, 5, 6, 7, 8, 9]
10 [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
10 [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
10 [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
11 [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
11 [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
11 [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]


In [21]:
import numpy as np

def coordinates_in_eigenbasis(A, v):
    """
    Compute the coordinates of vector v in the eigenbasis of matrix A.
    
    Parameters:
    - A: numpy array, square matrix.
    - v: numpy array, vector to represent in eigenbasis.
    
    Returns:
    - coords: numpy array, the coordinates of v in the eigenbasis of A.
    """
    # Step 1: Compute eigenvalues and eigenvectors of A
    eigvals, eigvecs = np.linalg.eig(A)
    
    # Step 2: Express v in the eigenbasis (coordinates in the basis of eigenvectors)
    # Project v onto the eigenvectors
    coords = np.dot(np.linalg.inv(eigvecs), v)
    
    return coords

# Example usage
A = np.array([[4, -1],
              [-1, 4]])

v = np.array([2, 1])

coords = coordinates_in_eigenbasis(A, v)
print("Coordinates of v in the eigenbasis of A:", coords)


Coordinates of v in the eigenbasis of A: [0.70710678 2.12132034]


In [22]:
def matrix_coordinates_in_eigenbasis(A, B):
    """
    Compute the coordinates of vector v in the eigenbasis of matrix A.
    
    Parameters:
    - A: numpy array, square matrix.
    - v: numpy array, vector to represent in eigenbasis.
    
    Returns:
    - coords: numpy array, the coordinates of v in the eigenbasis of A.
    """
    # Step 1: Compute eigenvalues and eigenvectors of A
    eigvals, eigvecs = np.linalg.eig(A)
    
    # Step 2: Express v in the eigenbasis (coordinates in the basis of eigenvectors)
    # Project v onto the eigenvectors

    n = len(eigvals)

    coords = np.zeros((n,n))

    for i in range(len(eigvals)):
        for j in range(len(eigvals)):
            left = eigvecs[:,i]
            rihgt = eigvecs[:,j]
            coords[i,j] = np.inner(np.conjugate(left),B.dot(rihgt))

    return coords

In [23]:
num_q = 4 # N # IMPORTANT

# Generate a random complex matrix
A = np.random.rand(2**num_q, 2**num_q) + 1j * np.random.rand(2**num_q, 2**num_q)
# Make it Hermitian
Hmat = (A + A.conj().T) / 2

In [24]:
plus=np.array([1,1])*(np.sqrt(2)/2)
psi = np.kron(plus,plus)
psi = np.kron(psi,plus)
psi = np.kron(psi,plus)
print(psi)
coordinates_in_eigenbasis(result, psi)
# print(matrix_coordinates_in_eigenbasis(result, Hmat))

[0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25
 0.25 0.25]


array([ 1.00000000e+00,  2.77555756e-16, -2.08166817e-16, -6.93889390e-17,
       -7.66747776e-16,  3.19189120e-16, -3.05311332e-16, -5.55111512e-17,
        2.49800181e-16, -2.22044605e-16,  1.80411242e-16, -7.63278329e-16,
       -2.42861287e-17, -2.08166817e-17,  7.07767178e-16,  2.49800181e-16])

In [25]:
def H_ZZ(num_q):
    box = [I for _ in range(num_q)]
    result = np.zeros((2**num_q, 2**num_q))

    for i in range(num_q):
        box = [I for _ in range(num_q)]
        box[i] = Z
        if i == num_q-1:
            box[0] = Z
        else:
            box[i+1] = Z    
        result += continuous_kronecker_prod(box)

    EIGS,EIGVS=np.linalg.eigh(-0.5*result)

    return EIGS,EIGVS